In [1]:
from datasets import load_dataset
import os

# Load dataset
ds = load_dataset("Renicames/turkish-law-chatbot")

print(ds)
print(ds["train"].column_names)

# Split original train into train + validation
split_data = ds["train"].train_test_split(
    test_size=0.1,
    seed=42,
    shuffle=True
)

train_ds = split_data["train"]
val_ds = split_data["test"]

# Use original test split if it exists
test_ds = ds["test"] if "test" in ds else None

# Create local folder
save_dir = "data/renicames_turkish_law_chatbot"
os.makedirs(save_dir, exist_ok=True)

# Save as JSONL
train_ds.to_json(f"{save_dir}/train.jsonl", force_ascii=False)
val_ds.to_json(f"{save_dir}/validation.jsonl", force_ascii=False)

if test_ds is not None:
    test_ds.to_json(f"{save_dir}/test.jsonl", force_ascii=False)

print("Saved files to:", save_dir)
print("Train size:", len(train_ds))
print("Validation size:", len(val_ds))
if test_ds is not None:
    print("Test size:", len(test_ds))

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/Renicames/turkish-law-chatbot/resolve/dd3b448409c6ef3ed97d4633d76097f0a81d78e4/turkish-law-chatbot.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since Renicames/turkish-law-chatbot couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\MEfe\.cache\huggingface\datasets\Renicames___turkish-law-chatbot\default\0.0.0\dd3b448409c6ef3ed97d4633d76097f0a81d78e4 (last modified on Wed May  6 23:20:28 2026).


DatasetDict({
    train: Dataset({
        features: ['Soru', 'Cevap'],
        num_rows: 13354
    })
    test: Dataset({
        features: ['Soru', 'Cevap'],
        num_rows: 1500
    })
})
['Soru', 'Cevap']


Creating json from Arrow format:   0%|          | 0/13 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Saved files to: data/renicames_turkish_law_chatbot
Train size: 12018
Validation size: 1336
Test size: 1500


In [2]:
from datasets import load_dataset
import os
import pandas as pd

ds = load_dataset("Renicames/turkish-law-chatbot")
print(ds)
print(ds["train"][0])  # See the column names and structure

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/Renicames/turkish-law-chatbot/resolve/dd3b448409c6ef3ed97d4633d76097f0a81d78e4/turkish-law-chatbot.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since Renicames/turkish-law-chatbot couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\MEfe\.cache\huggingface\datasets\Renicames___turkish-law-chatbot\default\0.0.0\dd3b448409c6ef3ed97d4633d76097f0a81d78e4 (last modified on Wed May  6 23:20:28 2026).


DatasetDict({
    train: Dataset({
        features: ['Soru', 'Cevap'],
        num_rows: 13354
    })
    test: Dataset({
        features: ['Soru', 'Cevap'],
        num_rows: 1500
    })
})
{'Soru': "Anayasa madde 1'e göre, türkiye'nin devlet şekli nedir", 'Cevap': "Anayasa madde 1'e göre, türkiye'nin devlet şekli cumhuriyettir. bu madde, türkiye'nin yönetim biçiminin halkın egemenliğine dayandığını ve bu yönetim biçiminin cumhuriyet olduğunu belirler. cumhuriyet, halkın kendi kendini yönetme biçimi olarak kabul edilir ve türkiye cumhuriyeti'nin temel yönetim ilkesi olarak anayasal güvence altına alınmıştır."}


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch

# ── Load model ──────────────────────────────────────────────────────────────
model_Qwen = "Qwen/Qwen2.5-1.5B-Instruct"
Qwen_tokenizer = AutoTokenizer.from_pretrained(model_Qwen)
Qwen_model = AutoModelForCausalLM.from_pretrained(
    model_Qwen,
    torch_dtype="auto",
    device_map="auto"
)

# ── Load dataset ─────────────────────────────────────────────────────────────
ds = load_dataset("Renicames/turkish-law-chatbot")
split = ds["train"]  # or "test" if available

# ── Inference function ────────────────────────────────────────────────────────
def ask_Qwen(question: str, system_prompt: str = "Sen Türk hukuku konusunda uzman bir asistansın.") -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    text = Qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = Qwen_tokenizer([text], return_tensors="pt").to(Qwen_model.device)
    with torch.no_grad():
        generated_ids = Qwen_model.generate(**inputs, max_new_tokens=512)
    generated_ids = [
        out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)
    ]
    return Qwen_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# ── Evaluate on N samples ─────────────────────────────────────────────────────
N = 40  # start small to test quickly
Qwen_results = []
question_list = split.select(range(N))

for i, sample in enumerate(question_list):
    question = sample["Soru"]   # ← adjust column name if needed
    reference = sample["Cevap"]       # ← adjust column name if needed

    prediction = ask_Qwen(question)

    Qwen_results.append({
        "question": question,
        "reference": reference,
        "prediction": prediction
    })
    
    qwen_quest_and_preds = f"Q: {question}\nPred: {prediction}\nRef: {reference}\n"
    
    print(f"[{i+1}/{N}] Q: {question[:80]}...")
    print(f"       Pred: {prediction[:120]}...\n")

# ── (Optional) Compute ROUGE scores ──────────────────────────────────────────
# pip install rouge-score
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
scores = [scorer.score(r["reference"], r["prediction"]) for r in Qwen_results]

avg_r1 = sum(s["rouge1"].fmeasure for s in scores) / len(scores)
avg_rL = sum(s["rougeL"].fmeasure for s in scores) / len(scores)

qwen_results = {
    "results": Qwen_results,
    "scores": scores,
    "avg_rouge1": avg_r1,
    "avg_rougeL": avg_rL,
}

row_df = pd.DataFrame(qwen_results)
file_exists = os.path.isfile("qwen_results.csv")
row_df.to_csv("qwen_results.csv", mode='w', index=False, encoding='utf-8-sig')

print(f"\nAvg ROUGE-1: {avg_r1:.4f}")
print(f"Avg ROUGE-L: {avg_rL:.4f}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/Renicames/turkish-law-chatbot/resolve/dd3b448409c6ef3ed97d4633d76097f0a81d78e4/turkish-law-chatbot.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since Renicames/turkish-law-chatbot couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\MEfe\.cache\huggingface\datasets\Renicames___turkish-law-chatbot\default\0.0.0\dd3b448409c6ef3ed97d4633d76097f0a81d78e4 (last modified on Wed May  6 23:20:28 2026).


[1/40] Q: Anayasa madde 1'e göre, türkiye'nin devlet şekli nedir...
       Pred: Türkiye'nin devlet şekli, Ulusal Anayasası (UYK) tarafından belirlenen Anayasa Madde 1'ü çerçevesinde tanımlanır. Bu ana...

[2/40] Q: Anayasa madde 1'de belirtilen cumhuriyetin tanımı nedir...
       Pred: Anayasa Madde 1, Türkiye Cumhuriyeti'nin anayasası ve idari yapısı hakkında genel kuralların ve öncü noktaları yer almak...

[3/40] Q: Anayasa madde 1, cumhuriyetin ilan edilmesini neden önemli görmüştür...
       Pred: Anayasa madde 1, cumhuriyetin anayasası olarak genellikle kurumsal ve sosyal durumları hakkında bilgi veren bir anayasad...

[4/40] Q: Anayasa madde 1, cumhuriyetin hangi tarihte ilan edildiğini belirtir mi...
       Pred: Anayasa madde 1'nin ilk kullanımının ve ilanı ne tarihinde gerçekleştiğine dair kesin bilgiye sahip değilim. Bu konuda g...

[5/40] Q: Anayasa madde 1'e göre, cumhuriyetin temel özellikleri nelerdir...
       Pred: Anayasa Madde 1'nin temel özellikleri şunlardır:

1. C

In [4]:
# ── Load model ──────────────────────────────────────────────────────────────
from huggingface_hub import login

login()

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


model_Llama = "meta-llama/Llama-3.2-1B-Instruct"
Llama_tokenizer = AutoTokenizer.from_pretrained(model_Llama,token=True)
Llama_model = AutoModelForCausalLM.from_pretrained(
    model_Llama,
    torch_dtype="auto",
    device_map="auto"
)

# ── Load dataset ─────────────────────────────────────────────────────────────

Llama_results = []


# ── Inference function ────────────────────────────────────────────────────────
def ask_Llama(question: str, system_prompt: str = "Sen Türk hukuku konusunda uzman bir asistansın.") -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    text = Llama_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = Llama_tokenizer([text], return_tensors="pt").to(Llama_model.device)
    with torch.no_grad():
        generated_ids = Llama_model.generate(**inputs, max_new_tokens=512)
    generated_ids = [
        out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)
    ]
    return Llama_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# ── Evaluate on N samples ─────────────────────────────────────────────────────

for i, sample in enumerate(question_list):
    question = sample["Soru"]   # ← adjust column name if needed
    reference = sample["Cevap"]       # ← adjust column name if needed

    prediction = ask_Llama(question)

    Llama_results.append({
        "question": question,
        "reference": reference,
        "prediction": prediction
    })
    
    llama_quest_and_preds = f"Q: {question}\nPred: {prediction}\nRef: {reference}\n"
    
    print(f"[{i+1}/{N}] Q: {question[:80]}...")
    print(f"       Pred: {prediction[:120]}...\n")

# ── (Optional) Compute ROUGE scores ──────────────────────────────────────────
# pip install rouge-score
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
scores = [scorer.score(r["reference"], r["prediction"]) for r in Llama_results]

avg_r1 = sum(s["rouge1"].fmeasure for s in scores) / len(scores)
avg_rL = sum(s["rougeL"].fmeasure for s in scores) / len(scores)


llama_results = {
    "results": Llama_results,
    "scores": scores,
    "avg_rouge1": avg_r1,
    "avg_rougeL": avg_rL,
}

row_df = pd.DataFrame(llama_results)
file_exists = os.path.isfile("llama_results.csv")
row_df.to_csv("llama_results.csv", mode='w', index=False, encoding='utf-8-sig')

print(f"\nAvg ROUGE-1: {avg_r1:.4f}")
print(f"Avg ROUGE-L: {avg_rL:.4f}")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[1/40] Q: Anayasa madde 1'e göre, türkiye'nin devlet şekli nedir...
       Pred: Anayasa, Türkiye'nin devlet şekli olarak "demokrasi" olarak tanımlanır....



[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[2/40] Q: Anayasa madde 1'de belirtilen cumhuriyetin tanımı nedir...
       Pred: Anayasa maddesi 1, "Cumhuriyetin, milletvevati olarak definedanın esasen, milletvevati olarak definedanın esasen, millet...



...

[37/40] Q: Anayasa madde 4'e göre, anayasa değişikliği tekliflerinin sınırları nelerdir...
       Pred: Anayasa maddeleri, 4. maddede değişiklikler için düzenlenmelidir....



[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[38/40] Q: Anayasa madde 5'e göre, devletin temel amaçları nelerdir...
       Pred: Anayasa, 5. maddede following şekilde düzenlenmiştir:

- Devletin temel amacı, milletin welfareini, demokrasi, anayasal ...



[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[39/40] Q: Anayasa madde 5, devletin görevlerini nasıl tanımlar...
       Pred: Anayasa 5, Türk Anayasası'nı oluşturan 5 maddede devletin görevlerini tanımlar. İşte Anayasa 5'inde tanımlanan görevler:...

[40/40] Q: Anayasa madde 5'e göre, devletin vatandaşlarına karşı sorumlulukları nelerdir...
       Pred: Anayasa maddeleri, devletin vatandaşlarına çeşitli sorumlulukları bulunmaktadır. İşte Anayasa'nın 5. maddesine göre devl...


Avg ROUGE-1: 0.1704
Avg ROUGE-L: 0.1324
